# Notebook 15 – Before vs After Preprocessing


**Dataset:** A reproducible synthetic telecom customer dataset. It contains missing values, extreme values, duplicate rows, categorical columns, and an imbalanced target. It is generated locally, so no download is required.

> Important: “After” means the output of the illustrative cleaning steps below. Real projects must choose treatments using domain knowledge and training data only.

## 1. Imports and reproducible raw dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1200
age = rng.normal(38, 12, n).clip(18, 80).round()
tenure = rng.integers(1, 73, n)
monthly_charges = (rng.normal(70, 22, n) + tenure * 0.08).clip(15, 160).round(2)
support_calls = rng.poisson(1.8, n)
contract = rng.choice(["Month-to-month", "One year", "Two year"], n, p=[0.58, 0.25, 0.17])
internet = rng.choice(["Fiber", "DSL", "No"], n, p=[0.48, 0.37, 0.15])
payment = rng.choice(["Card", "Bank transfer", "Electronic check"], n)
region = rng.choice(["North", "South", "East", "West"], n)
# Create an imbalanced target with plausible relationships
logit = -2.0 + 0.75*(contract == "Month-to-month") + 0.35*(support_calls >= 4) - 0.025*tenure + 0.012*(monthly_charges-70)
prob = 1/(1+np.exp(-logit))
churn = rng.binomial(1, prob)
df_raw = pd.DataFrame({
    "customer_id": [f"C{i:05d}" for i in range(n)],
    "age": age, "tenure_months": tenure, "monthly_charges": monthly_charges,
    "support_calls": support_calls, "contract": contract, "internet_service": internet,
    "payment_method": payment, "region": region, "churn": churn
})
# Introduce missingness, outliers, and exact duplicates for teaching
for col, frac in {"age":0.06, "monthly_charges":0.04, "contract":0.03, "payment_method":0.05}.items():
    idx = rng.choice(df_raw.index, int(n*frac), replace=False)
    df_raw.loc[idx, col] = np.nan
outlier_idx = rng.choice(df_raw.index, 8, replace=False)
df_raw.loc[outlier_idx, "monthly_charges"] = rng.uniform(350, 600, len(outlier_idx))
df_raw = pd.concat([df_raw, df_raw.sample(12, random_state=7)], ignore_index=True)
print("Raw dataset:", df_raw.shape)
df_raw.head()

## 2. Create a cleaned comparison dataset

Rules used for this demonstration:
- Remove exact duplicate rows.
- Impute numeric missing values with the median.
- Impute categorical missing values with the most frequent category.
- Cap the extreme `monthly_charges` values at the 1st and 99th percentiles.
- Keep `customer_id` as an identifier for auditing, but exclude it from model features.
- Do not modify the target `churn`.

In [ ]:
df_after = df_raw.drop_duplicates().copy()

numeric_cols = df_after.select_dtypes(include="number").columns.drop("churn")
categorical_cols = df_after.select_dtypes(include=["object", "category"]).columns.drop("customer_id")

num_imputer = SimpleImputer(strategy="median")
df_after[numeric_cols] = num_imputer.fit_transform(df_after[numeric_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
df_after[categorical_cols] = cat_imputer.fit_transform(df_after[categorical_cols])

low, high = df_after["monthly_charges"].quantile([0.01, 0.99])
df_after["monthly_charges"] = df_after["monthly_charges"].clip(lower=low, upper=high)

print("Before:", df_raw.shape, "| After:", df_after.shape)
df_after.head()

## 3. Shape and feature count

In [ ]:
shape_comparison = pd.DataFrame({
    "Before": [df_raw.shape[0], df_raw.shape[1], df_raw.shape[1]-2],
    "After": [df_after.shape[0], df_after.shape[1], df_after.shape[1]-2]
}, index=["Rows", "Columns", "Columns excluding ID and target"])
display(shape_comparison)

**Interpretation prompts:** Which dimension changed after duplicate removal? Why does imputing values usually not change shape? Why is the ID excluded from model features?

## 4. Data types

In [ ]:
dtype_comparison = pd.DataFrame({
    "Before": df_raw.dtypes.astype(str),
    "After": df_after.dtypes.astype(str)
})
display(dtype_comparison)
print("Data types changed:", (dtype_comparison["Before"] != dtype_comparison["After"]).sum())

Check whether numeric columns were accidentally converted to `object`, and whether categorical columns have the expected type.

## 5. Missing values

In [ ]:
missing_comparison = pd.DataFrame({
    "Before count": df_raw.isna().sum(),
    "After count": df_after.isna().sum(),
})
missing_comparison["Before %"] = (missing_comparison["Before count"] / len(df_raw) * 100).round(2)
missing_comparison["After %"] = (missing_comparison["After count"] / len(df_after) * 100).round(2)
display(missing_comparison)
missing_comparison[["Before count", "After count"]].plot(kind="bar", figsize=(11,4), title="Missing values: before vs after")
plt.ylabel("Missing cells"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 6. Duplicate records

In [ ]:
duplicate_comparison = pd.Series({
    "Before": int(df_raw.duplicated().sum()),
    "After": int(df_after.duplicated().sum())
}, name="Exact duplicate rows")
display(duplicate_comparison)

## 7. Outlier detection with the IQR rule

In [ ]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return int(((series < lower) | (series > upper)).sum())

outlier_comparison = pd.DataFrame({
    "Before": {c: iqr_outlier_count(df_raw[c].dropna()) for c in ["age","tenure_months","monthly_charges","support_calls"]},
    "After": {c: iqr_outlier_count(df_after[c]) for c in ["age","tenure_months","monthly_charges","support_calls"]}
})
display(outlier_comparison)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.boxplot(y=df_raw["monthly_charges"], ax=axes[0]); axes[0].set_title("Before: monthly charges")
sns.boxplot(y=df_after["monthly_charges"], ax=axes[1]); axes[1].set_title("After: monthly charges")
plt.tight_layout(); plt.show()

**Caution:** IQR flags unusual observations; it does not prove they are errors. Investigate whether extreme values are valid customers, data-entry errors, or rare business cases before treating them.

## 8. Numerical distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,8))
for ax, col in zip(axes.flat, ["age","tenure_months","monthly_charges","support_calls"]):
    sns.histplot(df_raw[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Before: {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12,8))
for ax, col in zip(axes.flat, ["age","tenure_months","monthly_charges","support_calls"]):
    sns.histplot(df_after[col], kde=True, ax=ax, color="darkorange")
    ax.set_title(f"After: {col}")
plt.tight_layout(); plt.show()

Compare skewness, spread, peaks, and tails. Median imputation can create a visible pile-up at the median; capping reduces the influence of extreme values.

## 9. Categorical values

In [ ]:
for col in ["contract","internet_service","payment_method","region"]:
    print(f"\\n{col} – before")
    display(df_raw[col].value_counts(dropna=False).to_frame("count"))
    print(f"{col} – after")
    display(df_after[col].value_counts(dropna=False).to_frame("count"))

## 10. Numerical ranges

In [ ]:
range_comparison = pd.DataFrame({
    "Before min": df_raw[["age","tenure_months","monthly_charges","support_calls"]].min(),
    "Before max": df_raw[["age","tenure_months","monthly_charges","support_calls"]].max(),
    "After min": df_after[["age","tenure_months","monthly_charges","support_calls"]].min(),
    "After max": df_after[["age","tenure_months","monthly_charges","support_calls"]].max()
})
display(range_comparison)

## 11. Target distribution and class imbalance

In [ ]:
target_counts = pd.DataFrame({
    "Before": df_raw["churn"].value_counts(normalize=True),
    "After": df_after["churn"].value_counts(normalize=True)
}).fillna(0).rename(index={0:"No churn",1:"Churn"})
display((target_counts*100).round(2))
target_counts.plot(kind="bar", figsize=(7,4), title="Target class proportions")
plt.ylabel("Proportion"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 12. Final comparison summary

In [ ]:
summary = pd.DataFrame({
    "Before": [
        *df_raw.shape, int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum()),
        int(df_raw.drop(columns=["customer_id","churn"]).shape[1])
    ],
    "After": [
        *df_after.shape, int(df_after.isna().sum().sum()), int(df_after.duplicated().sum()),
        int(df_after.drop(columns=["customer_id","churn"]).shape[1])
    ]
}, index=["Rows","Columns","Missing cells","Duplicate rows","Raw feature count (excluding ID/target)"])
display(summary)

## 13. Explain what changed and why

Complete this table in your own words:

| Area | What changed? | Why was it changed? | Possible risk |
|---|---|---|---|
| Shape | | | |
| Data types | | | |
| Missing values | | | |
| Duplicates | | | |
| Outliers | | | |
| Distributions | | | |
| Categories | | | |
| Numerical ranges | | | |
| Feature count | | | |

## 14. Reflection
1. Which preprocessing step changed the dataset shape?
2. Did median imputation make the data more accurate? Explain the limitation.
3. Why should outliers not always be removed?
4. Why must the target column not be imputed or transformed blindly?
5. Which comparison would you include in a preprocessing report, and why?